In [5]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import h5py
from tqdm.auto import tqdm

# ==================================================================================
# CUSTOM DATASET CLASS (FIXED)
# ==================================================================================
class EEGTextMetaDataset(Dataset):
    def __init__(self, eeg_dir, metadata_dir, tokenizer, 
                 color_map_file, object_map_file, 
                 max_length=64):
        
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.eeg_data_list = []
        self.index_map = []
        self.metadata_list = []

        # -------------------------
        # 1. Load EEG files (Initialization logic retained)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        for subj_idx, path in enumerate(eeg_files):
            eeg = np.load(path, mmap_mode='r')
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        print(f"Found {len(eeg_files)} EEG files → Total samples: {total_samples}")

        # -------------------------
        # 2. Load Metadata JSONs (Initialization logic retained)
        # -------------------------
        metadata_files = sorted(
            [os.path.join(dp, f)
             for dp, dn, filenames in os.walk(metadata_dir)
             for f in filenames if f.endswith(".json")]
        )
        
        if not metadata_files:
            raise FileNotFoundError(f"No metadata JSON files found in {metadata_dir}")

        for fpath in metadata_files:
            try:
                with open(fpath, 'r', encoding='utf-8') as f:
                    meta = json.load(f)

                caption_data = meta.get("caption")
                if isinstance(caption_data, dict):
                    meta['caption'] = caption_data.get("text", "")
                elif isinstance(caption_data, str):
                    meta['caption'] = caption_data
                else:
                    meta['caption'] = ""

                if "visual_attributes" not in meta:
                    meta["visual_attributes"] = {"major_colors": []}
                if "semantic_features" not in meta:
                    meta["semantic_features"] = {"objects": []}

                self.metadata_list.append(meta)

            except Exception as e:
                print(f"Warning: Skipping {fpath}: {e}")
                continue

        base_count = len(self.metadata_list)
        print(f"Loaded {base_count} metadata JSON files")

        num_subjects = len(self.eeg_data_list)
        self.metadata_repeated = self.metadata_list * num_subjects

        # -------------------------
        # 3. Load Mappings
        # -------------------------
        try:
            with open(color_map_file, 'r') as f:
                self.id_to_color = {int(k): v for k, v in json.load(f).items()}
                self.color_to_id = {v: k for k, v in self.id_to_color.items()}
                
            with open(object_map_file, 'r') as f:
                self.id_to_object = {int(k): v for k, v in json.load(f).items()}
                self.object_to_id = {v: k for k, v in self.id_to_object.items()}
                
        except Exception as e:
            raise RuntimeError(f"Failed to load mapping files: {e}")

        print(f"Vocabularies: {len(self.color_to_id)} colors, {len(self.object_to_id)} objects")

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        # 1. Get EEG
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        
        # 2. Get Metadata & Caption
        meta = self.metadata_repeated[idx]
        caption = meta.get("caption", "")

        # 3. Tokenize Text
        tokenized = self.tokenizer(
            caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        input_ids = tokenized['input_ids'].squeeze(0)

        # 4. Encode Metadata (Multi-Label)
        
        # A. Colors (Multi-Hot) - FIX APPLIED HERE
        color_multi_hot = torch.zeros(len(self.color_to_id), dtype=torch.float32)
        raw_colors = meta.get("visual_attributes", {}).get("major_colors", [])
        
        for c_item in raw_colors:
            # --- FIX: Safely extract color name, accommodating both dict and string formats ---
            if isinstance(c_item, dict):
                c_name = c_item.get("color")
            elif isinstance(c_item, str):
                c_name = c_item
            else:
                c_name = None

            if c_name and c_name in self.color_to_id:
                c_id = self.color_to_id[c_name]
                color_multi_hot[c_id] = 1.0

        # B. Objects (Multi-Hot)
        object_multi_hot = torch.zeros(len(self.object_to_id), dtype=torch.float32)
        raw_objects = meta.get("semantic_features", {}).get("objects", [])
        
        for o_name in raw_objects:
            if o_name in self.object_to_id:
                o_id = self.object_to_id[o_name]
                object_multi_hot[o_id] = 1.0

        # C. Concatenate [9 colors, 6 objects] -> 15 dim vector
        metadata_tensor = torch.cat((color_multi_hot, object_multi_hot))
        
        return eeg_tensor, input_ids, metadata_tensor

In [6]:
# ==================================================================================
# MAIN SCRIPT
# ==================================================================================

# --- Configuration (Updated for new Dataset) ---
HDF5_FILE = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
EEG_DIR = "/home/poorna/data/preprocessed_eeg"
METADATA_DIR = "/home/poorna/data/metadata_final_complete" 
TOKENIZER_PATH = "/home/poorna/models/bert-base-uncased"
COLOR_MAP_FILE = "/home/poorna/data/color_id_to_name_blip.json"
OBJECT_MAP_FILE = "/home/poorna/data/object_id_to_name_blip.json"

# --- Execution ---
if __name__ == "__main__":
    print(f"Initializing dataset from {METADATA_DIR}...")
    
    tokenizer = BertTokenizer.from_pretrained(TOKENIZER_PATH)

    dataset = EEGTextMetaDataset(
        eeg_dir=EEG_DIR,
        metadata_dir=METADATA_DIR,
        tokenizer=tokenizer,
        color_map_file=COLOR_MAP_FILE,
        object_map_file=OBJECT_MAP_FILE
    )

    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    n_samples = len(dataset)

    if n_samples == 0:
        print("Error: Dataset is empty.")
    else:
        # Dry run to get shapes
        sample_eeg, sample_input_ids, sample_meta = dataset[0] # This line now works
        
        print(f"\nCreating HDF5 file: {HDF5_FILE}")
        print(f"Total Samples: {n_samples}")
        print(f"Metadata Vector Shape: {sample_meta.shape} (Expected [15])")

        with h5py.File(HDF5_FILE, "w") as f:
            # Create datasets
            eeg_ds = f.create_dataset("eeg", shape=(n_samples, *sample_eeg.shape), dtype="float32")
            tokens_ds = f.create_dataset("input_ids", shape=(n_samples, *sample_input_ids.shape), dtype="int64")
            meta_ds = f.create_dataset("metadata", shape=(n_samples, *sample_meta.shape), dtype="float32")

            print("Writing data to HDF5 file...")
            try:
                for idx, (eeg, txt, meta) in enumerate(tqdm(loader, desc="Saving")):
                    eeg_ds[idx] = eeg.squeeze(0).numpy()
                    tokens_ds[idx] = txt.squeeze(0).numpy()
                    meta_ds[idx] = meta.squeeze(0).numpy()
                
                print(f"\nSuccess! Saved to {HDF5_FILE}")

            except Exception as e:
                print(f"\nCRITICAL ERROR during HDF5 writing at index {idx}: {e}")

Initializing dataset from /home/poorna/data/metadata_final_complete...
Found 20 EEG files → Total samples: 28000
Loaded 1400 metadata JSON files
Vocabularies: 9 colors, 6 objects

Creating HDF5 file: /home/poorna/data/eeg_dataset_1400_multilabel.h5
Total Samples: 28000
Metadata Vector Shape: torch.Size([15]) (Expected [15])
Writing data to HDF5 file...


Saving:   0%|          | 0/28000 [00:00<?, ?it/s]


Success! Saved to /home/poorna/data/eeg_dataset_1400_multilabel.h5


In [7]:
import h5py
import torch

with h5py.File("/home/poorna/data/eeg_dataset_1400_multilabel.h5", "r") as f:
    print(list(f.keys()))  # ['eeg', 'input_ids', 'metadata']
    eeg_sample = torch.tensor(f["eeg"][0])         # first sample EEG tensor
    tokens_sample = torch.tensor(f["input_ids"][0])
    meta_sample = torch.tensor(f["metadata"][0])

print(eeg_sample.shape, tokens_sample.shape, meta_sample.shape)

['eeg', 'input_ids', 'metadata']
torch.Size([62, 400]) torch.Size([64]) torch.Size([15])


In [8]:
import h5py

HDF5_FILE = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"

with h5py.File(HDF5_FILE, "r") as f:
    print("\nDatasets in file:", list(f.keys()))

    for name in f.keys():
        dset = f[name]
        print(f"{name}: shape={dset.shape}, dtype={dset.dtype}")

    # Optional: verify a few entries
    sample_idx = 0
    print("\nSample check:")
    print("EEG sample:", f["eeg"][sample_idx])
    print("Tokens sample:", f["input_ids"][sample_idx])
    print("Metadata sample:", f["metadata"][sample_idx])


Datasets in file: ['eeg', 'input_ids', 'metadata']
eeg: shape=(28000, 62, 400), dtype=float32
input_ids: shape=(28000, 64), dtype=int64
metadata: shape=(28000, 15), dtype=float32

Sample check:
EEG sample: [[-0.74070895 -0.77031434 -0.79470426 ...  0.42319107  0.28547502
   0.2032696 ]
 [-0.9918483  -1.0222337  -1.0500526  ...  0.5762788   0.42724377
   0.32933134]
 [-1.038139   -1.060072   -1.0697869  ...  0.44921234  0.326793
   0.2891903 ]
 ...
 [ 0.0523313   0.09942307  0.11695372 ...  0.40212336  0.5708321
   0.6923711 ]
 [ 0.5686282   0.5012047   0.4228295  ... -0.3007532  -0.1018642
  -0.00143331]
 [ 1.3704914   1.3944668   1.3755077  ... -0.73535657 -0.71439075
  -0.69585663]]
Tokens sample: [ 101 1037 2103 2007 4206 3121 1998 1037 2395  102    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 